In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install rank_bm25 faiss-cpu sentence-transformers -q

import torch
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import faiss

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH  = f"{DRIVE_BASE}/pubmedqa_filtered.json"
OUTPUT_DIR = f"{DRIVE_BASE}/biomistral_activation_patching_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus: {len(corpus)} samples")

In [ ]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)
model = model.to("cuda")
model.eval()
for param in model.parameters():
    param.requires_grad = False

n_layers    = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size  = model.config.vocab_size

print(f"Model: {n_layers} layers, hidden {hidden_size}, vocab {vocab_size}")

In [ ]:
documents, doc_metadata = [], []
for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({"pubid": sample["pubid"], "label": sample["label"]})

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(
    documents, batch_size=32, show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready: {index.ntotal} documents indexed")

In [ ]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k  = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "score": scores[i]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "score": float(s)}
            for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    combined = {}
    for rank, r in enumerate(bm25_retrieve(query, k=k*2)):
        combined[r["abstract"]] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    for rank, r in enumerate(faiss_retrieve(query, k=k*2)):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1/(rrf_k+rank+1)
        else:
            combined[key] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    return [v["meta"] for v in sorted(
        combined.values(), key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context = "\n\n".join(f"[{i+1}] {r['abstract']}"
                          for i, r in enumerate(results))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

def build_suppressed_prompt(query):
    return f"Question: {query}\nAnswer:"

print("Prompt functions ready.")

In [ ]:
def run_patched_pass(query, rag_hidden, patch_layer):
    sup_prompt = build_suppressed_prompt(query)
    inputs = tokenizer(
        sup_prompt, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")

    rag_last = rag_hidden[-1, :].to("cuda")
    hooks = []

    def make_hook(li):
        def hook(module, inp, output):
            if li == patch_layer:
                if isinstance(output, tuple):
                    hs = output.clone()
                    if hs.dim() == 3:
                        hs[0, -1:, :] = rag_last.to(hs.dtype)
                    else:
                        hs[0, -1, :] = rag_last.to(hs.dtype)
                        return hs
                    return (hs,) + output[1:]
                else:
                    # transformers 5.x: output is raw tensor (1, seq_len, hidden)
                    hs = output.clone()
                    hs[0, -1:, :] = rag_last.to(hs.dtype)
                    return hs
        return hook

    for i, block in enumerate(model.model.layers):
        hooks.append(block.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        outputs = model(**inputs)

    for h in hooks:
        h.remove()

    return torch.softmax(outputs.logits[0, -1, :].float(), dim=-1)

In [ ]:
N_SAMPLES = 30

YES_IDS = list(set(
    tid for t in ["yes", "Yes"]
    for tid in tokenizer.encode(t, add_special_tokens=False)
))
NO_IDS = list(set(
    tid for t in ["no", "No"]
    for tid in tokenizer.encode(t, add_special_tokens=False)
))

def get_final_probs(prompt):
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.softmax(outputs.logits[0, -1, :].float(), dim=-1)

def get_answer_probs(probs):
    p_yes = sum(probs[t].item() for t in YES_IDS if t < vocab_size)
    p_no  = sum(probs[t].item() for t in NO_IDS  if t < vocab_size)
    return p_yes, p_no

def kl_div(p, q):
    return float(torch.sum(
        p * (torch.log(p + 1e-10) - torch.log(q + 1e-10))
    ).item())

def save_rag_hidden(query, patch_layer):
    """Run RAG pass, return hidden state at patch_layer.
    BioMistral hooks return 2D (seq_len, hidden_size) — no batch dim."""
    prompt = build_rag_prompt(query)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")
    saved, hooks = {}, []

    def make_hook(li):
        def hook(module, inp, output):
            saved[li] = output[0].detach().squeeze(0) if output[0].dim() == 3 \
                        else output[0].detach()
        return hook

    for i, block in enumerate(model.model.layers):
        hooks.append(block.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    return saved[patch_layer]

def run_patched_pass(query, rag_hidden, patch_layer):
    """
    Suppressed pass with RAG hidden state injected at patch_layer.
    Only the last token position is patched.
    BioMistral: output[0] is 2D (seq_len, hidden_size), index as hs[-1:, :]
    """
    sup_prompt = build_suppressed_prompt(query)
    inputs = tokenizer(
        sup_prompt, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")

    rag_last = rag_hidden[-1:, :]
    hooks = []

    def make_hook(li):
        def hook(module, inp, output):
            if li == patch_layer:
                hs = output[0].clone()
                hs[-1:, :] = rag_last.to(hs.dtype)
                return (hs,) + output[1:]
        return hook

    for i, block in enumerate(model.model.layers):
        hooks.append(block.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        outputs = model(**inputs)

    for h in hooks:
        h.remove()

    return torch.softmax(outputs.logits[0, -1, :].float(), dim=-1)

print("Patching functions ready.")
print(f"YES token IDs: {YES_IDS} | NO token IDs: {NO_IDS}")

In [ ]:
patching_samples = [s for s in corpus if s["label"] == "no"][:N_SAMPLES]
print(f"Selected {len(patching_samples)} no-label samples")

In [ ]:
def run_patched_pass(query, rag_hidden, patch_layer):
    sup_prompt = build_suppressed_prompt(query)
    inputs = tokenizer(
        sup_prompt, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")

    # rag_hidden is (seq_len, hidden) — take last token as scalar index
    rag_last = rag_hidden[-1, :].to("cuda")  # (hidden_size,)
    hooks = []

    def make_hook(li):
        def hook(module, inp, output):
            if li == patch_layer:
                # output is (1, seq_len, hidden) — raw tensor in transformers 5.x
                hs = output.clone()
                hs[0, -1, :] = rag_last.to(hs.dtype)
                return hs  # return tensor, not tuple
        return hook

    for i, block in enumerate(model.model.layers):
        hooks.append(block.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        outputs = model(**inputs)

    for h in hooks:
        h.remove()

    return torch.softmax(outputs.logits[0, -1, :].float(), dim=-1)

print("run_patched_pass redefined")

In [ ]:
PATCH_LAYER = 24

print(f"Running benchmark patch at layer {PATCH_LAYER} (T4 peak)...\n")
benchmark_results = []

for sample in tqdm(patching_samples, desc=f"Layer {PATCH_LAYER}"):
    query = sample["query"]

    rag_hidden = save_rag_hidden(query, PATCH_LAYER)
    p_rag      = get_final_probs(build_rag_prompt(query))
    p_sup      = get_final_probs(build_suppressed_prompt(query))
    p_patched  = run_patched_pass(query, rag_hidden, PATCH_LAYER)

    kl_rag_sup     = kl_div(p_rag, p_sup)
    kl_rag_patched = kl_div(p_rag, p_patched)
    recovery       = (kl_rag_sup - kl_rag_patched) / (kl_rag_sup + 1e-8)

    p_yes_rag, p_no_rag = get_answer_probs(p_rag)
    p_yes_sup, p_no_sup = get_answer_probs(p_sup)
    p_yes_pat, p_no_pat = get_answer_probs(p_patched)

    benchmark_results.append({
        "pubid":             sample["pubid"],
        "query":             sample["query"][:60],
        "kl_rag_vs_sup":     round(kl_rag_sup, 4),
        "kl_rag_vs_patched": round(kl_rag_patched, 4),
        "recovery_pct":      round(float(recovery), 4),
        "patching_helped":   bool(kl_rag_patched < kl_rag_sup),
        "p_yes_rag":         round(p_yes_rag, 4),
        "p_yes_sup":         round(p_yes_sup, 4),
        "p_yes_pat":         round(p_yes_pat, 4),
        "p_no_rag":          round(p_no_rag, 4),
        "p_no_sup":          round(p_no_sup, 4),
        "p_no_pat":          round(p_no_pat, 4),
    })
    torch.cuda.empty_cache()

n_helped      = sum(1 for r in benchmark_results if r["patching_helped"])
mean_kl_sup   = np.mean([r["kl_rag_vs_sup"]     for r in benchmark_results])
mean_kl_patch = np.mean([r["kl_rag_vs_patched"]  for r in benchmark_results])
mean_recovery = np.mean([r["recovery_pct"]        for r in benchmark_results])

print(f"\n{'='*55}")
print(f"BENCHMARK PATCH — Layer {PATCH_LAYER}")
print(f"{'='*55}")
print(f"Patching helped:           {n_helped}/{N_SAMPLES} ({n_helped/N_SAMPLES:.1%})")
print(f"Mean KL(RAG || sup):       {mean_kl_sup:.4f}")
print(f"Mean KL(RAG || patched):   {mean_kl_patch:.4f}")
print(f"Mean recovery:             {mean_recovery:+.1%}")
print(f"BioMedLM layer 28 ref:     99.2% recovery, 30/30")

In [ ]:
SWEEP_LAYERS = list(range(32))
sweep_results = {}

print(f"Running full {len(SWEEP_LAYERS)}-layer sweep on {N_SAMPLES} samples...\n")

for sweep_layer in SWEEP_LAYERS:
    recoveries = []

    for sample in tqdm(patching_samples, desc=f"Layer {sweep_layer:2d}", leave=False):
        query = sample["query"]

        rag_hidden = save_rag_hidden(query, sweep_layer)
        p_rag      = get_final_probs(build_rag_prompt(query))
        p_sup      = get_final_probs(build_suppressed_prompt(query))
        p_patched  = run_patched_pass(query, rag_hidden, sweep_layer)

        kl_sup  = kl_div(p_rag, p_sup)
        kl_pat  = kl_div(p_rag, p_patched)
        rec     = (kl_sup - kl_pat) / (kl_sup + 1e-8)
        recoveries.append(rec)

        torch.cuda.empty_cache()

    sweep_results[sweep_layer] = {
        "mean_recovery": round(float(np.mean(recoveries)), 4),
        "pct_helped":    round(sum(1 for r in recoveries if r > 0) / len(recoveries), 3)
    }

print("\nFull layer sweep results:")
print(f"{'Layer':<8} {'Mean Recovery':<20} {'% Helped'}")
print("-" * 40)
for layer in SWEEP_LAYERS:
    res = sweep_results[layer]
    print(f"{layer:<8} {res['mean_recovery']:+.1%}{'':>10} {res['pct_helped']:.1%}")

output_path = os.path.join(OUTPUT_DIR, "activation_patching_results.json")
with open(output_path, "w") as f:
    json.dump({
        "model_id":   MODEL_ID,
        "patch_layer_benchmark": PATCH_LAYER,
        "n_samples":  N_SAMPLES,
        "label_filter": "no",
        "benchmark":  {
            "mean_kl_rag_vs_sup":     round(float(mean_kl_sup),   4),
            "mean_kl_rag_vs_patched": round(float(mean_kl_patch),  4),
            "mean_recovery_pct":      round(float(mean_recovery),  4),
            "n_helped":               n_helped,
        },
        "layer_sweep": sweep_results,
        "benchmark_samples": benchmark_results,
    }, f, indent=2)

print(f"\nSaved to {output_path}")
print("10_biomistral_activation_patching: complete")